# 02 · API·이벤트·중복 방지

**핵심 실행 12분** · 9주차 API 계약과 10주차 운영 기록을 사용자 단위 실험으로 연결합니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 격리된 실습 API | status=ok와 실험 설정 버전이 보입니다 |
| 3 | 두 버전의 실제 API 응답 | A와 B의 구성은 다르지만 정책 기간과 비용은 같습니다 |
| 4 | 같은 요청의 재전송 | 동일 요청 재전송은 True, 내용 변경 재사용은 409입니다 |
| 5 | 사용자 피드백과 분모 | 사용자 2명의 결과이며 WAIT입니다 |
| 6 | 이벤트 원본과 운영 지표 | Prometheus가 읽을 수 있는 지표 텍스트가 출력됩니다 |
| 7 | 긴급 중단 스위치 | 원래 B였던 사용자에게 A를 제공하고, 혼합 결과 분석은 INVALID입니다 |
| 8 | API 계약의 잘못된 입력 | 422가 반환되고 Notebook의 임시 DB가 정리됩니다 |

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 격리된 실습 API

TestClient는 실제 FastAPI 경로·검증·SQLite 저장을 호출합니다. TCP 네트워크나 실제 고객 트래픽을 재현한 것은 아닙니다.

In [ ]:
from tempfile import TemporaryDirectory
from fastapi.testclient import TestClient
from week12.lab.service import create_app
temp = TemporaryDirectory(prefix="w12-notebook-")
client = TestClient(create_app(Path(temp.name)/"events.sqlite3"))
print(client.get("/health").json())

**확인과 해석:** status=ok와 실험 설정 버전이 보입니다.

**직접 설명:** HTTP 계약 검증과 운영 환경 부하 검증은 무엇이 다른가요?

## 코드 셀 3 · 두 버전의 실제 API 응답

서버가 버전을 정합니다. 사용자가 마음대로 B를 고르면 비교 집단이 달라집니다.

In [ ]:
ids = {v: next(f"u-{i}" for i in range(100) if assign(f"u-{i}")==v) for v in ["A","B"]}
responses = {}
for variant, uid in ids.items():
    payload = {"visitor_id":uid,"request_id":f"req-{variant}","topic":"exchange","provider":"template"}
    response = client.post("/api/answer", json=payload)
    assert response.status_code == 200
    responses[variant] = response.json()
    print(variant, response.json()["answer"])
print("정책 템플릿 실행: 여기서는 모델을 호출하지 않았습니다.")

**확인과 해석:** A와 B의 구성은 다르지만 정책 기간과 비용은 같습니다.

**직접 설명:** 응답 모양이 달라도 반드시 같아야 할 사실은 무엇인가요?

## 코드 셀 4 · 같은 요청의 재전송

재시도는 흔합니다. 같은 request_id를 그대로 재전송하면 결과만 재사용하며 기록을 늘리지 않습니다.

In [ ]:
same = {"visitor_id":ids["A"],"request_id":"req-A","topic":"exchange","provider":"template"}
print(client.post("/api/answer", json=same).json()["replayed"])
assert len(client.get("/api/events").json()) == 2
conflict = client.post("/api/answer", json={**same,"topic":"return"})
print("서로 다른 요청에 ID 재사용:", conflict.status_code)
assert conflict.status_code == 409

**확인과 해석:** 동일 요청 재전송은 True, 내용 변경 재사용은 409입니다.

**직접 설명:** 중복 이벤트가 주로 B에서 발생하면 배정 비율은 어떻게 보일까요?

## 코드 셀 5 · 사용자 피드백과 분모

해결 확인은 사용자 피드백이며 모델 judge 점수가 아닙니다. 미응답은 확인 없음으로 남깁니다.

In [ ]:
print(client.post("/api/feedback",json={"request_id":"req-B","resolved":True}).json())
summary = client.get("/api/summary").json()
display(pd.DataFrame(summary["arms"]).T)
print(summary["decision"], summary["reasons"])

**확인과 해석:** 사용자 2명의 결과이며 WAIT입니다. A의 미응답은 분모에서 사라지지 않습니다.

**직접 설명:** 두 번의 요청에서 B의 비율이 높다고 배포할 수 있나요?

## 코드 셀 6 · 이벤트 원본과 운영 지표

동일 요청에 ID·할당·실제 제공 버전·오류·지연을 남깁니다. metrics는 요청 수, 분석은 사용자 수를 사용합니다.

In [ ]:
display(pd.DataFrame(client.get("/api/events").json())[["request_id","visitor_id","assigned_variant","served_variant","status","latency_ms","resolved"]])
print(client.get("/metrics").text)

**확인과 해석:** Prometheus가 읽을 수 있는 지표 텍스트가 출력됩니다. Prometheus 서버 자체를 띄운 것은 아닙니다.

**직접 설명:** p95가 좋아졌다는 운영 관찰만으로 B의 사업 효과를 주장할 수 있나요?

## 코드 셀 7 · 긴급 중단 스위치

운영 위험이 있으면 새 요청을 A로 되돌립니다. 중단 뒤 기록을 기존 A/B 효과 분석에 합치지 않습니다.

In [ ]:
print(client.post("/api/control",json={"enabled":False}).json())
rolled = client.post("/api/answer",json={"visitor_id":ids["B"],"request_id":"req-rollback","topic":"exchange","provider":"template"}).json()
print("원래 배정 / 실제 제공:", rolled["assigned_variant"], rolled["variant"])
print(client.get("/api/summary").json())

**확인과 해석:** 원래 B였던 사용자에게 A를 제공하고, 혼합 결과 분석은 INVALID입니다.

**직접 설명:** 운영 복구가 성공했다는 사실과 실험이 유효하다는 사실은 같은가요?

## 코드 셀 8 · API 계약의 잘못된 입력

새로운 모델을 붙이기 전에 요청·응답 계약을 유지해야 합니다.

In [ ]:
bad = client.post("/api/answer", json={"visitor_id":"name@example.com","request_id":"invalid","topic":"exchange"})
print("잘못된 실습 ID:",bad.status_code)
assert bad.status_code == 422
client.close()
temp.cleanup()
print("Notebook 임시 DB 정리 완료")

**확인과 해석:** 422가 반환되고 Notebook의 임시 DB가 정리됩니다.

**직접 설명:** 서비스의 입력 검증이 실험 신뢰성과 개인정보 보호에 어떤 역할을 하나요?

## 복습과 다음 단계

**실제 화면 실행:** VS Code 터미널에서 `python -m uvicorn week12.lab.service:app --host 127.0.0.1 --port 8012`를 실행하고 http://127.0.0.1:8012 를 엽니다. 요청 → 피드백 → 결과 확인 → 중단 스위치 순서로 시연합니다. 이 로컬 제어 API에는 운영용 인증이 없으므로 외부 공개 서버로 실행하지 않습니다.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
